In [1]:
import urllib
import feedparser
from unstructured.partition.pdf import partition_pdf
from io import BytesIO
import time
import os

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
# from qdrant_client import QdrantClient
# from qdrant_client.http.models import Distance, VectorParams

from langchain_core.prompts import PromptTemplate

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain.messages import HumanMessage,SystemMessage

c:\Users\louay\miniconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\louay\AppData\Local\Temp\ipykernel_3488\1160930089.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


## Document loading

In [2]:
params = {
    "search_query": "all:artificial intelligence",
    "start": 0,
    "max_results": 3,
}
url = f"https://export.arxiv.org/api/query?{urllib.parse.urlencode(params, safe=':')}"
data = urllib.request.urlopen(url).read().decode('utf-8')
d: feedparser.util.FeedParserDict = feedparser.parse(data)

In [3]:
pdfs = {}
for entry in d.entries:
    pdf_url = next(link.href for link in entry.links if link.type == "application/pdf")
    id = entry.id.split("/")[-1]
    title = entry.title
    pdfs[id] = {"url": pdf_url,
                "title": entry.get("title", ""),
                "author": entry.get("author", ""),
                "tags": entry.get("tags", [])}
    

In [4]:
documents = []

for id, data in pdfs.items():
    url = data["url"]

    t0 = time.time()
    try:
        with urllib.request.urlopen(url) as response:
            pdf_bytes = BytesIO(response.read())
        elements = partition_pdf(file=pdf_bytes, languages=["eng"], strategy="fast")
        doc = "\n\n".join([element.text for element in elements if element.category not in ["Header", "Footer", "PageBreak"]])
        documents.append(Document(page_content=doc, metadata=data | {"id": id}))
    except:
        print(f"Failed to read file {id} at {url}.")

    # ArXiv rate limit is 3 seconds.
    t1 = time.time()
    if (t1 - t0) < 3:
        time.sleep(3 - t1 + t0)


## Document chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=500, chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

Document(metadata={'url': 'https://arxiv.org/pdf/2110.01831v1', 'title': 'The Artificial Scientist: Logicist, Emergentist, and Universalist Approaches to Artificial General Intelligence', 'author': 'Yoshihiro Maruyama', 'tags': [{'term': 'cs.AI', 'scheme': 'http://arxiv.org/schemas/atom', 'label': None}], 'id': '2110.01831v1'}, page_content='1 2 0 2\n\nt c O 5\n\n] I\n\nA . s c [\n\n1 v 1 3 8 1 0 . 0 1 1 2 : v i X r a\n\nThe Artiﬁcial Scientist: Logicist, Emergentist, and Universalist Approaches to Artiﬁcial General Intelligence⋆\n\nMichael Timothy Bennett and Yoshihiro Maruyama\n\nSchool of Computing, Australian National University, Canberra, Australia michael.bennett@anu.edu.au and yoshihiro.maruyama@anu.edu.au\n\nAbstract. We attempt to deﬁne what is necessary to construct an Arti- ﬁcial Scientist, explore and evaluate several approaches to artiﬁcial gen- eral intelligence (AGI) which may facilitate this, conclude that a uniﬁed or hybrid approach is necessary and explore two theorie

In [6]:
# emb_dim = 64
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
) 

C:\Users\louay\AppData\Local\Temp\ipykernel_3488\4188157443.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9362.49it/s]


## Vector storage

In [8]:
url = "http://localhost:6333"
store = QdrantVectorStore.from_documents(
    docs,
    embedder,
    url=url,
    prefer_grpc=False,
    collection_name="qdrant_docs",
)

## Query

In [9]:
query = "Give me an AI paper."
retriever = store.as_retriever(search_type = "similarity", search_kwargs = {"k": 4})
context = retriever.invoke(query)

In [10]:
# Inspecting the first line of each chunk.
for i, doc in enumerate(context):
    print(f"Document {i + 1}: {doc.page_content.split("\n")[0]}")

Document 1: 1 2 0 2
Document 2: 4.2 Emergentist
Document 3: 2 2 0 2
Document 4: 2. Deﬁning Creative Problem Solving in AI


## Generation

In [11]:
HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
SYSTEM_PROMPT = """You are a helpful assistant that answers user queries based primarily on the context information provided to you.
This contexts contains factual information that pertain to the subject. If the context is insufficient to answer, state your inability to answer."""

In [21]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",  # let Hugging Face choose the best provider for you
)

chat_model = ChatHuggingFace(llm=llm)

In [22]:
prompt_template = "The user asked: {user_query} Use the following information to answer accurately:\n{context}."
template = PromptTemplate.from_template(prompt_template)
formatted_prompt = template.format(user_query = query, context = "\n".join([document.page_content for document in context]))

In [23]:
messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(formatted_prompt)]

In [24]:
ai_msg = chat_model.invoke(messages)
ai_msg.content

'Based on the context provided, here is a relevant AI paper:\n\nTitle: The Artiﬁcial Scientist: Logicist, Emergentist, and Universalist Approaches to Artiﬁcial General Intelligence\nAuthors: Michael Timothy Bennett and Yoshihiro Maruyama\nJournal: School of Computing, Australian National University, Canberra, Australia\nEmails: michael.bennett@anu.edu.au and yoshihiro.maruyama@anu.edu.au\nAbstract:\nThe paper defines necessary qualities for an Artiﬁcial Scientist, evaluates different approaches to achieving Artiﬁcial General Intelligence (AGI), and concludes that a unified or hybrid approach is required. The paper explores two theories that satisfy this requirement to some degree. The approaches discussed include emergentist and logicist methodologies.\n\nPlease note that the actual content of the paper is provided in the context, and this is just a citation based on the metadata given.'